# 04 — Double ML via dowhy + EconML
Fit CausalForestDML and run the full refutation suite.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.doubleml_estimator import run_doubleml, run_refutations
from src.viz import plot_refutation_table
from src.evaluation import evaluate_all
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_parquet("../data/simulation_observational.parquet")
feature_cols = [c for c in df.columns if c not in ["treatment", "outcome", "tau_true", "propensity"]]
df.shape

## Fit CausalForestDML

In [ ]:
dml_result = run_doubleml(df, feature_cols, n_estimators=300, seed=42)
ate_val = dml_result['ate']
ci_val = dml_result['ate_ci']
print(f'ATE = {ate_val:.4f} | 95% CI = {ci_val}')

## Per-user CATE estimates

In [ ]:
tau_dml = dml_result['tau_hat']
print(f'Mean CATE: {tau_dml.mean():.4f} | Std: {tau_dml.std():.4f}')

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.hist(tau_dml, bins=50, color='mediumseagreen', edgecolor='white')
ate_m = tau_dml.mean()
plt.axvline(ate_m, color='red', linestyle='--', label=f'ATE = {ate_m:.4f}')
plt.legend()
plt.title('CausalForestDML — CATE distribution')
plt.xlabel('tau_hat(X)')
plt.tight_layout()
plt.savefig('../results/figures/dml_cate_dist.png', bbox_inches='tight')
plt.show()

## Refutation tests

In [ ]:
refutation_results = run_refutations(
    df, feature_cols, dml_result,
    n_simulations=30,
    seed=42,
)
for rname, res in refutation_results.items():
    print(rname)
    print(res)

## Refutation table

In [ ]:
fig = plot_refutation_table(refutation_results, save=True)
fig.show()

## Metrics comparison (DoubleML vs meta-learners)

In [ ]:
import pickle
with open('../results/meta_learner_results.pkl', 'rb') as f:
    meta_results = pickle.load(f)

tau_hats_all = {name: res['tau_hat'] for name, res in meta_results.items()}
tau_hats_all['DoubleML'] = tau_dml

metrics = evaluate_all(df, tau_hats_all)
print(metrics.to_string())
metrics.to_csv('../results/all_model_metrics.csv')
import numpy as np
np.save('../results/dml_tau_hat.npy', tau_dml)
print('All results saved.')